# Phase 2 results: zero-shot and five-shot prompting

Reviews the output of `scripts/run_prompting.py` for one split (dev by default):

1. sanity checks (are all runs there and complete?)
2. main comparison table
3. five-shot gains and cross-lingual gaps
4. output-format reliability (malformed rate etc.)
5. per-emotion F1
6. label cardinality and empty predictions
7. paired bootstrap confidence intervals: which differences are real?
8. example errors
9. wording sensitivity (if `--sensitivity` was run)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from llm_project.notebook_utils import (
    load_summary,
    main_table,
    gain_and_gap,
    format_df,
    per_label_f1,
    predicted_distribution,
    empty_predictions_summary,
    load_preds,
    paired_bootstrap,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLIT = "dev"
ID_KEY = "alignment_id"
LABELS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

PRED_DIR = PROJECT_ROOT / "outputs" / "predictions" / "prompting" / SPLIT
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics" / "prompting" / SPLIT
pd.set_option("display.precision", 3)

summary = load_summary(METRICS_DIR)

## 1. Sanity checks
Expect 8 rows (2 models × 2 languages × 2 conditions), the full dev size in `n` (not 20 from the smoke test), and the same prompt fingerprint everywhere.

In [ ]:
print(f"{len(summary)} runs")
display(summary[["model", "role", "lang", "condition", "n"]])

fingerprints = {json.loads(p.read_text())["fingerprint"] for p in (METRICS_DIR).glob("*__default.json")}
print("prompt fingerprints:", fingerprints)

## 2. Main results table

In [ ]:
main = main_table(summary)
main

In [ ]:
ax = (
    summary.assign(run=summary["lang"] + " / " + summary["condition"]) 
    .pivot(index="run", columns="model", values="macro_f1")
    .plot.bar(figsize=(8, 4), rot=0)
)
ax.set_ylabel("macro-F1")
ax.set_title(f"Macro-F1 on {SPLIT}")
plt.tight_layout()
plt.show()

## 3. Five-shot gain and cross-lingual gap
Positive five-shot gain = demonstrations help. Positive EN−SV gap = the model is worse on Swedish.

In [ ]:
gain_gap = gain_and_gap(summary)
gain_gap

## 4. Output-format reliability
The spec asks for the malformed-output rate separately. `strict_json_rate` = whole output was exactly a JSON list; `wellformed_but_no_label_rate` = valid list but no permitted label (e.g. `[]`).

In [ ]:
format_df(summary)

## 5. Per-emotion F1

In [ ]:
from llm_project.notebook_utils import per_label_f1

per_label_f1 = per_label_f1(summary, METRICS_DIR, LABELS)
display(per_label_f1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
f1_only = per_label_f1.drop(columns="gold_support")
im = ax.imshow(f1_only.values, cmap="viridis", vmin=0, vmax=max(0.6, f1_only.values.max()))
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xticks(range(f1_only.shape[1]), ["\n".join(c) for c in f1_only.columns], fontsize=7)
for i in range(f1_only.shape[0]):
    for j in range(f1_only.shape[1]):
        ax.text(j, i, f"{f1_only.iat[i, j]:.2f}", ha="center", va="center", fontsize=7, color="w")
fig.colorbar(im, label="F1")
ax.set_title("Per-emotion F1")
plt.tight_layout()
plt.show()

## 6. Label cardinality, label distribution and empty predictions

In [ ]:
from llm_project.notebook_utils import load_preds

display(summary.set_index(["model", "lang", "condition"])[
    ["gold_label_cardinality", "pred_label_cardinality"]
])

In [ ]:
from llm_project.notebook_utils import predicted_distribution

dist = predicted_distribution(summary, PRED_DIR, LABELS)
display(dist)

In [ ]:
from llm_project.notebook_utils import empty_predictions_summary

for model, lang, condition, nempty, n, counts in empty_predictions_summary(summary, PRED_DIR):
    print(f"{model:28s} {lang} {condition:9s}: {nempty:4d}/{n} empty | gold of those: {counts}")

## 7. Paired bootstrap: which differences are real?

Resamples the dev texts 1,000 times. For a comparison A vs B, the *same* resampled texts are
used for both (paired, matched by alignment ID), which removes the noise from which texts were sampled.
If the 95% interval of the difference excludes 0, the difference is unlikely to be noise.

In [ ]:
from llm_project.notebook_utils import load_preds, paired_bootstrap

models = summary["model"].unique()
langs = summary["lang"].unique()
comparisons = []
for mo in models:
    for la in langs:
        comparisons.append(("five vs zero", (mo, la, "zero_shot"), (mo, la, "five_shot")))
    for co in ["zero_shot", "five_shot"]:
        other = [l for l in langs if l != "en"][0]
        comparisons.append((f"{other} vs en", (mo, "en", co), (mo, other, co)))
if len(models) == 2:
    for la in langs:
        for co in ["zero_shot", "five_shot"]:
            comparisons.append(
                (f"{models[1]} vs {models[0]}", (models[0], la, co), (models[1], la, co))
            )

rows = []
for name, a, b in comparisons:
    try:
        da = load_preds(a[0], a[1], a[2], PRED_DIR).set_index(ID_KEY)
        db = load_preds(b[0], b[1], b[2], PRED_DIR).set_index(ID_KEY)
    except Exception:
        continue
    d, lo, hi, n = paired_bootstrap(da, db, labels=LABELS)
    rows.append({"comparison": name, "A": f"{a[0]} / {a[1]} / {a[2]}", "B": f"{b[0]} / {b[1]} / {b[2]}", "n_paired": n, "macro_f1_diff (B-A)": d, "ci_low": lo, "ci_high": hi, "significant": not (lo <= 0 <= hi)})
boot = pd.DataFrame(rows)
display(boot)

## 8. Example errors
Read a sample of wrong predictions; use these for the qualitative error analysis.

In [ ]:
from llm_project.notebook_utils import show_errors

r0 = summary.iloc[0]
show_errors(r0["model"], [l for l in langs if l != "en"][0], "five_shot", PRED_DIR)

## 9. Wording sensitivity (dev only)
Only runs if `scripts/run_prompting.py --split dev --sensitivity` has finished.

In [ ]:
sens_path = METRICS_DIR / "summary_sensitivity.csv"
if sens_path.exists():
    sens = pd.read_csv(sens_path)
    table = sens.pivot_table(
        index=["model", "lang", "condition"], columns="variant", values="macro_f1"
    )
    table["range"] = table.max(axis=1) - table.min(axis=1)
    table["std"] = table.drop(columns="range").std(axis=1)
    display(table)
    display(
        sens.pivot_table(
            index=["model", "lang", "condition"], columns="variant", values="malformed_rate"
        )
    )
else:
    print("No sensitivity results yet: sbatch jobs/run_prompting.sh dev --sensitivity")